# A02-02 — Centrality Analysis
## Box House — Mapping the Spatial Hierarchy

**Project:** Box House — Graph-ML Assignment 02  
**Author:** Symon Kipkemei  
**Date:** 2026-05-19

---

Not all rooms play the same role in a building's circulation network. Some rooms are easy to reach from everywhere. Others are structurally indispensable — spaces that most occupants must pass through to get where they are going. This notebook maps that hierarchy.

Two metrics are applied to the Box House circulation graph. Closeness centrality identifies which room is the most accessible. Betweenness centrality identifies which room is the structural hub. Together they answer: which rooms has the building layout made important?

## 1. Import Libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color

e:\softwares-4\graph-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [2]:
print("This notebook requires topologicpy version 0.9.33 or newer.")
print(Helper.Version())

This notebook requires topologicpy version 0.9.33 or newer.
The version that you are using (0.9.33) is EQUAL TO the latest version available on PyPI.


## 3. Set Renderer
* Visual Studio Code: `"vscode"`
* Google Colab: `"colab"`
* Browser: `"browser"`

In [3]:
renderer = "vscode"

## 4. Load Geometry

In [4]:
objects = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-rooms.obj",
    selfMerge=True
)
print("Room objects:", objects)

Room objects: [<topologic_core.Cluster object at 0x000002D12067E270>]


In [5]:
doors   = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-doors.obj",
    selfMerge=True
)
windows = Topology.ByOBJPath(
    r"E:\softwares-4\graph-ml\assign-01-graphs\geometry\box-house-windows.obj",
    selfMerge=True
)

aperture_faces = []
for ap in doors:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["brown", "door"]))
        aperture_faces.append(f)
for ap in windows:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color", "type"], ["cyan", "window"]))
        aperture_faces.append(f)

print("Total apertures:", len(aperture_faces))

Total apertures: 36


## 5. Build CellComplex and Add Apertures

In [6]:
cells = Topology.Cells(objects[0])
cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)
cc = Topology.AddApertures(cc, aperture_faces, subTopologyType="face")
print("Cells:", len(cells))
print("Apertures registered:", len(aperture_faces))

Cells: 19
Apertures registered: 36


## 6. Build Circulation Graph

Same graph as A02-01: 39 nodes (room centroids), 40 edges (aperture-bearing shared walls). Interior circulation only — exterior apertures excluded.

In [7]:
g_circ = Graph.ByTopology(
    cc,
    direct=False,
    viaSharedApertures=True,
    toExteriorApertures=False
)
g_verts = Graph.Vertices(g_circ)
print("Vertices:", len(g_verts))
print("Edges:   ", len(Graph.Edges(g_circ)))

Vertices: 39
Edges:    40


## 7. Which Room Is Easiest to Reach?

Closeness centrality measures how shallow each room is within the network — how quickly an occupant can reach it from anywhere else in the building. A high score means few transitions are needed on average to arrive from any other room.

In spatial terms: the most integrated room is where movement naturally converges. If the building is well-organised, this should be a corridor, lobby, or shared space — not a private room at the end of a dead-end passage.

In [8]:
cc_values = Graph.ClosenessCentrality(g_circ, colorScale="thermal")

if cc_values:
    max_cc  = max(cc_values)
    min_cc  = min(cc_values)
    best_i  = cc_values.index(max_cc)
    worst_i = cc_values.index(min_cc)

    print(f"Closeness range: {min_cc:.4f} â†’ {max_cc:.4f}")
    print(f"Most  central room: index {best_i}  (score {max_cc:.4f})")
    print(f"Least central room: index {worst_i} (score {min_cc:.4f})")
    print()
    print(f"{'Index':>6}  {'Closeness':>10}")
    print("-" * 22)
    for i, val in enumerate(cc_values):
        marker = " â† most central" if i == best_i else (
                 " â† least central" if i == worst_i else "")
        print(f"{i:>6}  {val:>10.4f}{marker}")

Closeness range: 0.0964 â†’ 0.2043
Most  central room: index 1  (score 0.2043)
Least central room: index 4 (score 0.0964)

 Index   Closeness
----------------------
     0      0.1881
     1      0.2043 â† most central
     2      0.2000
     3      0.1473
     4      0.0964 â† least central
     5      0.1545
     6      0.1681
     7      0.1583
     8      0.1532
     9      0.1234
    10      0.1348
    11      0.1180
    12      0.1234
    13      0.1275
    14      0.1218
    15      0.1258
    16      0.1027
    17      0.1073
    18      0.1073
    19      0.1949
    20      0.1659
    21      0.1704
    22      0.2032
    23      0.1854
    24      0.1784
    25      0.1751
    26      0.1315
    27      0.1064
    28      0.1372
    29      0.1372
    30      0.1456
    31      0.1502
    32      0.1382
    33      0.1402
    34      0.1382
    35      0.1121
    36      0.1199
    37      0.1199
    38      0.1121


### Closeness: What the Result Reveals

**Room 1 is the most integrated space in the building** (score 0.2043). It is reachable more quickly from the rest of the network than any other room. On the thermal map it appears as the hottest node — the spatial centre of gravity of the Box House circulation network.

**Room 4 is the most isolated** (score 0.0964) — the coldest node. On average, reaching room 4 requires almost twice as many transitions as reaching room 1. It sits at the periphery of the aperture network.

The score range (0.10–0.20) shows a consistent hierarchy. The building does not have a single dominant centre, but room 1 is clearly its spatial focus.

In [9]:
# cc_color is written to each vertex by Graph.ClosenessCentrality
for e in Graph.Edges(g_circ):
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [2, "grey"]))

Topology.Show(
    cc, g_circ,
    faceOpacity=0.1,
    vertexSize=18,
    vertexColorKey="cc_color",
    edgeWidthKey="width",
    edgeColorKey="color",
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 8. Which Room Is the Structural Hub?

Betweenness centrality counts how often each room lies on the shortest path between two other rooms. A high score means most routes through the building pass through that space — it is structurally indispensable. Blocking or removing it would disrupt movement for many occupants.

This is the corridor question: which room is functioning as the building's spine whether or not it was designed as one?

In [10]:
bc_values = Graph.BetweennessCentrality(g_circ, normalize=True, colorScale="thermal")

if bc_values:
    max_bc  = max(bc_values)
    min_bc  = min(bc_values)
    best_i  = bc_values.index(max_bc)

    print(f"Betweenness range: {min_bc:.4f} â†’ {max_bc:.4f}")
    print(f"Highest betweenness (hub room): index {best_i}  (score {max_bc:.4f})")
    print()
    print(f"{'Index':>6}  {'Betweenness':>12}")
    print("-" * 24)
    for i, val in enumerate(bc_values):
        marker = " â† hub" if i == best_i else ""
        print(f"{i:>6}  {val:>12.4f}{marker}")

Betweenness range: 0.0000 â†’ 0.6522
Highest betweenness (hub room): index 1  (score 0.6522)

 Index   Betweenness
------------------------
     0        0.5676
     1        0.6522 â† hub
     2        0.5078
     3        0.1935
     4        0.0000
     5        0.3151
     6        0.3585
     7        0.1707
     8        0.0683
     9        0.0725
    10        0.1991
    11        0.1024
    12        0.0725
    13        0.0000
    14        0.0000
    15        0.0107
    16        0.0064
    17        0.0000
    18        0.0000
    19        0.5007
    20        0.2347
    21        0.3414
    22        0.5121
    23        0.3713
    24        0.1991
    25        0.1110
    26        0.1494
    27        0.0526
    28        0.1166
    29        0.1166
    30        0.0526
    31        0.2347
    32        0.0526
    33        0.0341
    34        0.0284
    35        0.0284
    36        0.0526
    37        0.0526
    38        0.0284


### Betweenness: What the Result Reveals

**Room 1 again scores highest** (0.6522) — it is both the most accessible and the most structurally critical room in the building. Over 65% of all shortest paths between room pairs pass through room 1. It is the undisputed hub of the Box House circulation network.

**Rooms 0 and 22** (0.5676 and 0.5121) also score high, forming a cluster of load-bearing spaces at the core of the network.

**Five rooms score zero betweenness** (4, 13, 14, 17, 18) — they carry no through-movement. These are terminal rooms: occupants arrive and leave back the same way. They are not on the path to anywhere else.

In [11]:
# bc_color is written to each vertex by Graph.BetweennessCentrality
Topology.Show(
    cc, g_circ,
    faceOpacity=0.1,
    vertexSize=18,
    vertexColorKey="bc_color",
    edgeWidthKey="width",
    edgeColorKey="color",
    backgroundColor="black",
    width=800,
    height=600,
    renderer=renderer
)

## 9. The Spatial Backbone

Sorting all rooms by closeness and cross-referencing betweenness identifies the rooms that matter most to the building's spatial structure.

**Rooms 1, 22, 2, 19, and 0** form the top tier on both metrics — high integration and high betweenness. These five rooms are the spatial backbone of the Box House. The building's circulation is organised around them.

**Rooms 4, 16, 17, and 18** sit at the bottom on both metrics. They are spatially peripheral — neither easy to reach nor structurally important. These are the building's most private, enclosed spaces.

In [12]:
if cc_values and bc_values:
    print(f"{'Index':>6}  {'Closeness':>10}  {'Betweenness':>12}")
    print("-" * 36)
    combined = sorted(
        zip(range(len(cc_values)), cc_values, bc_values),
        key=lambda x: x[1],
        reverse=True
    )
    for i, cc_val, bc_val in combined:
        print(f"{i:>6}  {cc_val:>10.4f}  {bc_val:>12.4f}")

 Index   Closeness   Betweenness
------------------------------------
     1      0.2043        0.6522
    22      0.2032        0.5121
     2      0.2000        0.5078
    19      0.1949        0.5007
     0      0.1881        0.5676
    23      0.1854        0.3713
    24      0.1784        0.1991
    25      0.1751        0.1110
    21      0.1704        0.3414
     6      0.1681        0.3585
    20      0.1659        0.2347
     7      0.1583        0.1707
     5      0.1545        0.3151
     8      0.1532        0.0683
    31      0.1502        0.2347
     3      0.1473        0.1935
    30      0.1456        0.0526
    33      0.1402        0.0341
    32      0.1382        0.0526
    34      0.1382        0.0284
    28      0.1372        0.1166
    29      0.1372        0.1166
    10      0.1348        0.1991
    26      0.1315        0.1494
    13      0.1275        0.0000
    15      0.1258        0.0107
     9      0.1234        0.0725
    12      0.1234        0.0725
    14